In [ ]:
import os
import pandas as pd

def merge_results(save_path1, region, label_list, data,
                  dim=0):

    all_dfs = []

    for label in label_list:

        csv_path = (
            save_path1 + "d" + str(dim) +
            "_" + label + region + ".csv"
        )

        if not os.path.exists(csv_path):
            print(f"找不到: {csv_path}")
            continue

        df_cell = pd.read_csv(csv_path)

        # 添加细胞及实验信息
        cell_info = data[
            data['label1'] == label
        ].iloc[0]

        df_cell['label1'] = label
        df_cell['Experiment_ID'] = cell_info['Experiment_ID']
        df_cell['Group'] = cell_info['Group']
        df_cell['Cell_ID'] = cell_info['Cell_ID']

        all_dfs.append(df_cell)

    if len(all_dfs) == 0:
        raise ValueError(f"{region}: 没有找到任何 CSV 文件")

    merged = pd.concat(all_dfs, ignore_index=True)

    # 检查读取到的细胞数量
    n_cells = merged['label1'].nunique()
    print(f"{region}: 成功读取 {n_cells} 个细胞")

    if n_cells != len(label_list):
        print(f"警告：预期 {len(label_list)} 个细胞，实际 {n_cells} 个")

    merged.to_csv(
        path + "d" + str(dim) +
        # 修改：加 tag
        "_merged_" + region + f"_{tag}.csv",
        index=False
    )

    print(f"合并完成：{len(merged)} 行")

    return merged

In [ ]:
save_path1 = pd_dir   # 修改

bd_s = merge_results(
    save_path1=save_path1,
    region='SNCA',
    label_list=s['label1'].unique(),
    data=s,
    dim=0
)

bd_u = merge_results(
    save_path1=save_path1,
    region='UBL3',
    label_list=u['label1'].unique(),
    data=u,
    dim=0
)

bd_c = merge_results(
    save_path1=save_path1,
    region='CAAX',
    label_list=c['label1'].unique(),
    data=c,
    dim=0
)

# 合并三组
bd = pd.concat([bd_s, bd_u, bd_c], ignore_index=True)

In [ ]:
from pandas.api.types import CategoricalDtype
#bd.columns = ['sample',	'Birth mean',	'Death mean',	'Persistence mean','Birth variance',	'Death variance',	'Persistence variance',
#             'dimension','group','n_comp','1% persistence','label1','Experiment','Group','cell']
bd.columns = [
    "sample",
    "dimension",
    "group",
    "n_total",
    "n_finite",
    "n_infinite",
    "birth_mean",
    "birth_median",
    "birth_q25",
    "birth_q75",
    "birth_iqr",
    "birth_variance",
    "birth_std",
    "death_mean",
    "death_median",
    "death_q25",
    "death_q75",
    "death_iqr",
    "death_variance",
    "death_std",
    "persistence_mean",
    "persistence_median",
    "persistence_q25",
    "persistence_q75",
    "persistence_iqr",
    "persistence_variance",
    "persistence_std",
    "persistence_99th_percentile",
    "total_persistence",
    "persistent_entropy",
    "label1",
    "Experiment",
    "Group",
    "Cell_ID"
]
#order = CategoricalDtype(categories=['PFF SNCA', 'NO PFF SNCA'], ordered=True)
order = CategoricalDtype(categories=['SNCA', 'UBL3', 'CAAX'], ordered=True) #修改点
bd['group'] = bd['group'].astype(order)
bd =bd.sort_values('group')
bd.head()

import re
def multi_replace(text):

    text = re.sub(r'^(SNCA\+UBL3).*', 'Syn-HiBiT+AAV-Myc-UBL3', text, flags=re.IGNORECASE) #AAV-Myc
    text = re.sub(r'^(SNCA\+CAAX).*', 'Syn-HiBiT+AAV-Myc-UBL3∆5', text, flags=re.IGNORECASE)

    #修改点
    text = re.sub(r'^(NO PFF UBL3).*', 'Syn-HiBiT+AAV-Myc-UBL3', text, flags=re.IGNORECASE) #AAV-Myc
    text = re.sub(r'^(NO PFF CAAX).*', 'Syn-HiBiT+AAV-Myc-UBL3∆5', text, flags=re.IGNORECASE)
    text = re.sub(r'^(NO PFF SNCA).*', 'Syn-HiBiT', text, flags=re.IGNORECASE)

    text = re.sub(r'^(UBL3).*', 'Syn-HiBiT+AAV-Myc-UBL3', text, flags=re.IGNORECASE) #AAV-Myc
    text = re.sub(r'^(CAAX).*', 'Syn-HiBiT+AAV-Myc-UBL3∆5', text, flags=re.IGNORECASE)
    text = re.sub(r'^(SNCA).*', 'Syn-HiBiT', text, flags=re.IGNORECASE)

    #sns
    #text = re.sub(r'^(PFF SNCA).*', 'PFF Syn', text, flags=re.IGNORECASE)
    #text = re.sub(r'^(NO PFF SNCA).*', 'NO PFF Syn', text, flags=re.IGNORECASE)
    #text = re.sub(r'^(No2|No4).*', 'no2|4', text)

    return text
bd['group1'] = bd['group'].apply(multi_replace)
bd

In [ ]:
import pandas as pd
import numpy as np
from itertools import combinations
from scipy import stats
from statsmodels.stats.multitest import multipletests

# 需要分析的指标
metrics = [
    "n_total", "n_finite", "n_infinite",
    "birth_mean", "birth_median", "birth_q25", "birth_q75", "birth_iqr", "birth_variance", "birth_std",
    "death_mean", "death_median", "death_q25", "death_q75", "death_iqr", "death_variance", "death_std",
    "persistence_mean", "persistence_median", "persistence_q25", "persistence_q75", "persistence_iqr",
    "persistence_variance", "persistence_std", "persistence_99th_percentile",
    "total_persistence", "persistent_entropy",
]

ALPHA = 0.05

# 实际生物学分组
groups = list(pd.unique(bd['Group']))
experiments = sorted(bd['Experiment'].unique())

print("Groups:", groups)
print("Experiments:", experiments)
print("Dimensions:", bd['dimension'].unique())

# 每次实验、每组，先计算10个细胞的平均值（实验为统计单位）
exp_means = (
    bd.groupby(['dimension', 'Experiment', 'Group'])[metrics]
      .mean()
      .reset_index()
)


def sig_stars(p):
    return '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'n.s.'


def group_ranking(means):
    """按均值从高到低排序，例如 'UBL3 > CAAX > SNCA'"""
    s = pd.Series(means).sort_values(ascending=False)
    return ' > '.join(s.index)


overall_results = []
pairwise_results = []
pairs = list(combinations(groups, 2))

# 分别分析 H0、H1
for dim in sorted(bd['dimension'].unique()):

    sub = exp_means[exp_means['dimension'] == dim]

    for feature in metrics:

        wide = sub.pivot(index='Experiment', columns='Group', values=feature)
        wide = wide.reindex(columns=groups).dropna()

        # 确保每个指标都有3次完整独立实验
        if wide.shape[0] != 3:
            print(f"Warning: {feature}, H{dim}, n={wide.shape[0]}")
            continue

        # ---------- 整体检验（Friedman）+ 各组均值与排序 ----------
        friedman_stat, friedman_p = stats.friedmanchisquare(*[wide[g].values for g in groups])
        grp_mean = {g: wide[g].mean() for g in groups}
        grp_sd   = {g: wide[g].std(ddof=1) for g in groups}

        row = {
            'dimension': dim,
            'feature': feature,
            'n_experiments': len(wide),
            'Friedman_stat': friedman_stat,
            'Friedman_p': friedman_p,
            'ranking_high_to_low': group_ranking(grp_mean),     # 方向：谁最高、谁最低
            'highest_group': max(grp_mean, key=grp_mean.get),
            'lowest_group': min(grp_mean, key=grp_mean.get),
        }
        for g in groups:
            row[f'mean_{g}'] = grp_mean[g]
            row[f'sd_{g}'] = grp_sd[g]
        overall_results.append(row)

        # ---------- 三组两两配对比较 ----------
        raw_p, rows = [], []
        for g1, g2 in pairs:
            x = wide[g1].values
            y = wide[g2].values
            d = x - y                                   # 每次实验的差值（g1 − g2）
            t_stat, p_value = stats.ttest_rel(x, y)
            raw_p.append(p_value)

            md = d.mean()
            if md > 0:
                higher, lower = g1, g2
            elif md < 0:
                higher, lower = g2, g1
            else:
                higher = lower = None

            # 有多少次实验与平均方向一致
            n_consistent = int(np.sum(np.sign(d) == np.sign(md))) if md != 0 else 0

            # 相对变化：以较低的组为参照
            ref = np.mean(wide[lower].values) if lower is not None else np.nan
            pct = (abs(md) / abs(ref) * 100) if (lower is not None and ref != 0) else np.nan

            # 配对差值的 95% CI（df = n − 1）
            se = d.std(ddof=1) / np.sqrt(len(d))
            tcrit = stats.t.ppf(0.975, df=len(d) - 1)

            r = {
                'dimension': dim,
                'feature': feature,
                'comparison': f'{g1} vs {g2}',
                f'mean_group1': x.mean(),
                f'mean_group2': y.mean(),
                'group1': g1,
                'group2': g2,
                'mean_difference(g1-g2)': md,
                'diff_CI95_low': md - tcrit * se,
                'diff_CI95_high': md + tcrit * se,
                'diff_per_experiment': np.round(d, 4).tolist(),
                'higher_group': higher,
                'lower_group': lower,
                'direction': f'{higher} > {lower}' if higher else 'equal',
                'consistent_experiments': f'{n_consistent}/{len(d)}',
                'percent_higher(%)': pct,
                't_stat': t_stat,
                'p_raw': p_value,
            }
            rows.append(r)

        # 同一个指标、同一个dimension内，对3次比较进行Holm校正
        adjusted_p = multipletests(raw_p, method='holm')[1]
        for r, p_adj in zip(rows, adjusted_p):
            r['p_Holm'] = p_adj
            r['sig'] = sig_stars(p_adj)
            # 结论：显著时给方向；不显著时只标注趋势
            if p_adj < ALPHA:
                r['conclusion'] = f"{r['direction']} ({r['sig']})"
            else:
                r['conclusion'] = f"n.s. (trend: {r['direction']})"
        pairwise_results.extend(rows)


# 转为DataFrame
overall_df = pd.DataFrame(overall_results)
pairwise_df = pd.DataFrame(pairwise_results)

print("\n=== Overall Friedman test + group ranking ===")
display(overall_df)

print("\n=== Pairwise paired t-test with direction ===")
display(pairwise_df)

# ---------- 只看显著结果，方便直接写进 Results ----------
sig_df = pairwise_df[pairwise_df['p_Holm'] < ALPHA][
    ['dimension', 'feature', 'direction', 'mean_difference(g1-g2)',
     'percent_higher(%)', 'consistent_experiments', 'p_Holm', 'sig']
].sort_values(['dimension', 'feature'])

print("\n=== Significant pairwise differences (Holm p < 0.05) ===")
display(sig_df)

# ---------- 方向汇总宽表：每个指标一行，每个比较一列 ----------
direction_wide = pairwise_df.pivot_table(
    index=['dimension', 'feature'], columns='comparison',
    values='conclusion', aggfunc='first'
).reset_index()
direction_wide = direction_wide.merge(
    overall_df[['dimension', 'feature', 'ranking_high_to_low', 'Friedman_p']],
    on=['dimension', 'feature'], how='left'
)

print("\n=== Direction summary ===")
display(direction_wide)

# 保存（path 为你前面定义的结果目录）
overall_df.to_csv(path + 'PD_stats_overall_direction.csv', index=False)
pairwise_df.to_csv(path + 'PD_stats_pairwise_direction.csv', index=False)
sig_df.to_csv(path + 'PD_stats_significant_direction.csv', index=False)
direction_wide.to_csv(path + 'PD_stats_direction_summary.csv', index=False)